# Topic analysis of app reviews: App and Year effects on topic prevalence

This notebook fits a topic model (LDA with covariate adjustment) to user reviews of six applications (Amaha, MindDoc, Replika, Woebot, Wysa, Youper), estimates how topic prevalence differs by application and year, and runs sensitivity analyses for the unequal number of reviews per application.

## Pipeline

1. **Data preparation**: reads the review file, removes applications with fewer than 100 reviews (Happify, Mindspa), maps the columns to a common schema and derives the review year.
2. **Topic model**: text preprocessing (NLTK stop words and lemmatisation), vocabulary selection, LDA (scikit-learn, K = 15) with App and Year as covariates.
3. **Effects on topic prevalence**: App + Year regression of the topic proportions using the method of composition (as in R `stm::estimateEffect`), with 95% intervals, joint Wald tests, pairwise contrasts, Benjamini–Hochberg correction and a "meaningful difference" rule (interval excludes 0, q < 0.05, difference ≥ 2 percentage points).
4. **Number of topics**: held-out perplexity, semantic coherence, exclusivity and topic diversity across K and seeds (as in `stm::searchK`).
5. **Sensitivity analyses**: applications weighted equally; without Replika; Replika down-sampled to the largest other application (three samples); Replika only (Year effects).
6. **Export**: all results in three files.

## Input

A CSV with one review per row and the columns `App`, `date_only` and `content` (optionally `title`). Set its path in the data-preparation cell (default `/content/body_and_covariates_only.csv`, for Google Colab).

## Output

| File | Contents |
|---|---|
| `results_main.xlsx` | Main topic model (settings, document and term counts, topics, representative reviews, all documents), App + Year effects, diagnostics for K |
| `results_sensitivity.xlsx` | All sensitivity analyses, stacked with an `Analysis` column, and comparisons with the main analysis |
| `figures.pdf` | All figures, one per page |

## Requirements

Python 3 with pandas, numpy, scipy, scikit-learn, nltk, matplotlib, patsy and openpyxl. Run the cells in order; the K diagnostics and sensitivity analyses take several minutes.


## Setup


In [ ]:
# patsy is needed for the effect estimation (usually pre-installed in Colab)
!pip install -q patsy

## Data preparation
Reads the reviews, removes Happify and Mindspa, maps the columns to a common schema and writes `reshaped.csv`.


In [ ]:
# Imports
import pandas as pd
from pathlib import Path
from typing import Optional, Dict, Any, List

def reshape_posts_explicit(
    in_path: str,
    out_path: str = "reshaped.csv",
    *,
    date_col: str = "date_only",
    app_col: str = "App",
    title_col: str = "title",
    content_col: str = "content",
    keep_original_date: bool = True,
    explain: bool = True,
    exclude_apps: tuple = ("Happify", "Mindspa"),  # removed before any other step
) -> pd.DataFrame:
    """
    Read CSV, explicitly map columns to a unified schema, derive year_quarter when needed,
    build combined_text, and write the result.

    OUTPUT SCHEMA:
      forum           ← App
      forum_original  ← App
      post_type       ← year_quarter (if present) else derived from date_only as 'YYYY-Qn' else ""
      original_title  ← title (or "")
      original_post   ← content (or "")
      combined_text   ← original_title + " " + original_post (normalized)
      ethnicity       ← passthrough if present else ""
      gender          ← passthrough if present else ""
      age_group       ← passthrough if present else ""
      countrycode     ← passthrough if present else ""
      (optional) date_only kept if present and keep_original_date=True
    """
    in_path = Path(in_path)
    out_path = Path(out_path)
    df = pd.read_csv(in_path, encoding="latin1")
    global EXCLUSION_COUNTS  # saved in Document_Flow of results_main.xlsx
    EXCLUSION_COUNTS = [("Reviews collected (all applications)", len(df))]

    # Remove the excluded applications first, so that every later step
    # (vocabulary, topic model, effects, sensitivity analyses) is based on the remaining applications only
    if exclude_apps and app_col in df.columns:
        _app = df[app_col].astype(str).str.strip()
        _drop = _app.isin(list(exclude_apps))
        print(f"Loaded {len(df)} reviews; removed {int(_drop.sum())} from "
              f"{', '.join(exclude_apps)}: " + _app[_drop].value_counts().to_dict().__repr__())
        for _a in exclude_apps:
            EXCLUSION_COUNTS.append((f"Excluded: {_a} (fewer than 100 reviews)", -int((_app == _a).sum())))
        df = df[~_drop].reset_index(drop=True)

    # --------- helpers ---------
    def pick_source(candidates: List[str]) -> Optional[str]:
        for c in candidates:
            if c in df.columns:
                return c
        return None

    def get_series(col_name: Optional[str], default: str = "") -> pd.Series:
        if col_name and col_name in df.columns:
            return df[col_name]
        return pd.Series([default] * len(df), index=df.index)

    # mapping rules
    mapping_plan: Dict[str, List[str]] = {
        "forum":          [app_col],
        "forum_original": [app_col],
        "post_type":      ["year_quarter"],  # derive from date_col if missing
        "original_title": [title_col],
        "original_post":  [content_col],
        # passthrough covariates (single-name lookups)
        "ethnicity":      ["ethnicity"],
        "gender":         ["gender"],
        "age_group":      ["age_group"],
        "countrycode":    ["countrycode"],
    }

    mapping_decisions: Dict[str, Dict[str, Any]] = {}

    # --- forum / forum_original
    src_forum = pick_source(mapping_plan["forum"])
    forum = get_series(src_forum)
    mapping_decisions["forum"] = {"used": src_forum, "fallback": None}
    forum_original = forum
    mapping_decisions["forum_original"] = {"used": src_forum, "fallback": None}

    # --- post_type (year_quarter or derive from date)
    src_post_type = pick_source(mapping_plan["post_type"])
    if src_post_type:
        post_type = get_series(src_post_type).astype(str)
        mapping_decisions["post_type"] = {"used": src_post_type, "fallback": None}
    else:
        if date_col in df.columns:
            dt = pd.to_datetime(df[date_col], errors="coerce", dayfirst=False)
            year = dt.dt.year
            qtr  = dt.dt.quarter
            post_type = pd.Series(
                [f"{int(y)}-Q{int(q)}" if pd.notna(d) else "" for d, y, q in zip(dt, year, qtr)],
                index=df.index
            )
            mapping_decisions["post_type"] = {"used": f"derived_from:{date_col}", "fallback": None}
        else:
            post_type = pd.Series([""] * len(df), index=df.index)
            mapping_decisions["post_type"] = {"used": None, "fallback": "empty"}

    # Calendar year (needed for the App + Year prevalence covariates)
    if date_col in df.columns:
        year_series = pd.to_datetime(df[date_col], errors="coerce", dayfirst=False).dt.year.astype("Int64")
        mapping_decisions["year"] = {"used": f"derived_from:{date_col}", "fallback": None}
    else:
        year_series = pd.Series(pd.NA, index=df.index, dtype="Int64")
        mapping_decisions["year"] = {"used": None, "fallback": "empty"}

    # --- original_title / original_post
    src_title = pick_source(mapping_plan["original_title"])
    original_title = get_series(src_title)
    mapping_decisions["original_title"] = {"used": src_title, "fallback": "" if not src_title else None}

    src_post = pick_source(mapping_plan["original_post"])
    original_post = get_series(src_post)
    mapping_decisions["original_post"] = {
        "used": src_post,
        "fallback": "" if not src_post else None,
        "preference_order": mapping_plan["original_post"],
    }

    # --- passthrough covariates
    extras = {}
    for c in ["ethnicity", "gender", "age_group", "countrycode"]:
        src = pick_source(mapping_plan[c])
        s = get_series(src).astype(str).fillna("")
        extras[c] = s
        mapping_decisions[c] = {"used": src, "fallback": "" if not src else None}

    # Normalize to str + clean
    forum          = forum.astype(str).fillna("")
    forum_original = forum_original.astype(str).fillna("")
    post_type      = post_type.astype(str).fillna("")
    original_title = original_title.astype(str).fillna("")
    original_post  = original_post.astype(str).fillna("")

    combined_text = (
        (original_title.str.strip() + " " + original_post.str.strip())
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    out = pd.DataFrame({
        "forum": forum,
        "forum_original": forum_original,
        "post_type": post_type,
        "year": year_series,
        "original_title": original_title,
        "original_post": original_post,
        "combined_text": combined_text,
        "ethnicity": extras["ethnicity"],
        "gender": extras["gender"],
        "age_group": extras["age_group"],
        "countrycode": extras["countrycode"],
    })

    if keep_original_date and (date_col in df.columns):
        out[date_col] = df[date_col]

    if explain:
        print("\n=== Column Mapping Summary ===")
        def _fmt(dec):
            if not isinstance(dec, dict):
                return str(dec)
            used = dec.get("used")
            fb   = dec.get("fallback")
            pref = dec.get("preference_order")
            bits = []
            if pref: bits.append(f"pref={pref}")
            bits.append(f"used={used!r}")
            if fb is not None: bits.append(f"fallback={fb!r}")
            return ", ".join(bits)
        for out_col in [
            "forum","forum_original","post_type","year","original_title","original_post",
            "combined_text","ethnicity","gender","age_group","countrycode"
        ]:
            if out_col == "combined_text":
                print(f"{out_col:16s} <- computed: original_title + ' ' + original_post (normalized)")
            else:
                print(f"{out_col:16s} <- {_fmt(mapping_decisions.get(out_col, {}))}")
        print("\nRows:", len(out))

    out.to_csv(out_path, index=False, encoding="utf-8")
    print(f"\u2705 Wrote: {out_path}")
    return out

# Example call:
reshape_posts_explicit(
    "/content/body_and_covariates_only.csv", "reshaped.csv",
    date_col="date_only", app_col="App", title_col="title", content_col="content",
    exclude_apps=("Happify", "Mindspa"),   # fewer than 100 reviews; excluded from the whole analysis
)

## Topic model
Preprocessing, vocabulary selection, LDA with covariate adjustment, and the tables and heatmap of the fitted model.


In [ ]:
# Streamlined STM Analysis — parameterized & documented (multi-covariate)
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from typing import Literal, Optional, Dict, Any, Tuple, List
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import sys, platform, scipy, matplotlib, sklearn     # for reporting package versions
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score
from scipy.stats import entropy
import nltk
from nltk.corpus import stopwords

def _ensure_nltk():
    try: nltk.data.find("corpora/stopwords")
    except LookupError: nltk.download("stopwords")
    try: nltk.data.find("corpora/wordnet")
    except LookupError: nltk.download("wordnet")

def _is_categorical(series: pd.Series) -> bool:
    # Any non-numeric column (including the pandas >= 3 "str" dtype) is categorical
    return not pd.api.types.is_numeric_dtype(series)

def build_covariate_matrix(
    df: pd.DataFrame,
    covariate_cols: List[str],
    *, scale_numeric: bool = False
) -> Tuple[Optional[np.ndarray], List[str], Dict[str, Any]]:
    """
    Build a design matrix from multiple covariates (categorical + numeric).
    Returns:
      X_cov  : ndarray (n_docs, n_features_cov) or None if no covariates
      cols   : list of expanded column names (after dummies)
      meta   : dict with keys:
               - 'categorical': list[str] original cat covariates
               - 'numeric'    : list[str] original numeric covariates
               - 'expanded_map': dict[orig_col -> list[expanded_col(s)]]
    """
    if not covariate_cols:
        return None, [], {"categorical": [], "numeric": [], "expanded_map": {}}

    cols = []
    parts = []
    expanded_map: Dict[str, List[str]] = {}
    categorical, numeric = [], []

    for col in covariate_cols:
        s = df[col] if col in df.columns else pd.Series(pd.NA, index=df.index, name=col)

        if _is_categorical(s):
            categorical.append(col)
            s = s.astype("string").fillna("MISSING")
            d = pd.get_dummies(s, prefix=col, drop_first=False)
            expanded_map[col] = d.columns.tolist()
            cols.extend(d.columns.tolist())
            parts.append(d.values)
        else:
            numeric.append(col)
            v = pd.to_numeric(s, errors="coerce").to_numpy()
            # impute to median
            med = np.nanmedian(v[np.isfinite(v)]) if np.any(np.isfinite(v)) else 0.0
            v = np.where(np.isfinite(v), v, med)
            if scale_numeric:
                mu = float(np.mean(v))
                sd = float(np.std(v))
                v = (v - mu) / (sd if sd != 0 else 1.0)
            v = v.reshape(-1, 1)
            expanded_map[col] = [col]
            cols.append(col)
            parts.append(v)

    X_cov = np.concatenate(parts, axis=1) if parts else None
    meta = {"categorical": categorical, "numeric": numeric, "expanded_map": expanded_map}
    return X_cov, cols, meta

class StreamlinedSTM:
    def __init__(self, n_topics:int, covariate_weight:float=0.4, alpha:float=0.1, beta:float=0.01, random_state:int=42):
        self.n_topics = n_topics
        self.covariate_weight = covariate_weight
        self.lda = LatentDirichletAllocation(
            n_components=n_topics, doc_topic_prior=alpha, topic_word_prior=beta,
            max_iter=50, learning_method="batch", random_state=random_state
        )
        # One-vs-rest logistic regression of the dominant topic on the covariates
        self.covariate_model = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=random_state))

    def fit(self, X, covariates=None, use_covariates:bool=True):
        self.doc_topic_dist_ = self.lda.fit_transform(X)
        self.lda_doc_topic_ = self.doc_topic_dist_.copy()  # keep unadjusted LDA proportions for effect estimation
        self.components_ = self.lda.components_
        if use_covariates and covariates is not None:
            topic_assignments = self.doc_topic_dist_.argmax(axis=1)
            self.covariate_model.fit(covariates, topic_assignments)
            # Place probabilities in the right topic columns (some topics may never be dominant)
            covariate_predictions = np.zeros_like(self.doc_topic_dist_)
            covariate_predictions[:, self.covariate_model.classes_] = self.covariate_model.predict_proba(covariates)
            adjusted = (1 - self.covariate_weight) * self.doc_topic_dist_ + self.covariate_weight * covariate_predictions
            self.doc_topic_dist_ = adjusted / adjusted.sum(axis=1, keepdims=True)
        return self

    def get_topic_words(self, feature_names, n_words:int=10) -> Dict[int, list]:
        out = {}
        for t, comp in enumerate(self.components_):
            idx = comp.argsort()[:-n_words-1:-1]
            out[t] = [feature_names[i] for i in idx]
        return out

def run_stm_analysis(
    *,
    input_csv: str = "reshaped.csv",
    output_xlsx: Optional[str] = None,
    output_heatmap_png: Optional[str] = None,

    # Content assembly
    input_mode: Literal["title_only", "post_only", "title_post"] = "post_only",

    # Modeling hyperparameters
    n_topics: int = 15,
    min_word_frequency: int = 5,
    max_doc_vocab_ratio: float = 0.05,
    use_covariates: bool = True,
    covariate_weight: float = 0.4,
    topic_concentration: float = 0.1,
    word_concentration: float = 0.01,
    top_samples_count: int = 20,
    random_state: int = 42,

    # Multi-covariate controls
    covariate_cols: Optional[List[str]] = None,
    heatmap_by: Optional[str] = None,
    scale_numeric_covariates: bool = False,

    # Column names in input_csv (kept for compatibility)
    col_forum: str = "forum",
    col_title: str = "original_title",
    col_post: str = "original_post",
    col_combined: str = "combined_text",
    col_post_type: str = "post_type",

    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run a streamlined topic model (LDA + optional covariate adjustment) and export:
      • An Excel workbook with config, topic summary, and per-topic top-k representatives.
      • A heatmap PNG of (chosen categorical) Covariate × Topic counts.

    INPUT FILE (CSV: `input_csv`)
    Expected columns (created by your earlier reshape step):
      - text columns: {col_title}, {col_post}, optional {col_combined}
      - optional categorical/numeric covariates selected via `covariate_cols`
    Notes:
      - Encoding: UTF-8 expected. Missing columns are tolerated where documented.
      - Rows with empty final text (after input_mode) are dropped.

    Returns: dict with keys:
      - 'excel': Path to Excel file
      - 'heatmap': Path to PNG file
      - 'df': DataFrame with assigned 'Topic' and 'Topic_Probability'
      - 'topic_words': dict[int -> list[str]] top words per topic
      - 'metrics': dict with perplexity, silhouette, covariate_consistency, topic_diversity
    """

    warnings.filterwarnings("ignore")
    _ensure_nltk()

    # ---------- File names ----------
    # None = write no file; tables are returned in res["sheets"] and the heatmap in res["figures"]

    if verbose:
        print("="*70)
        print("STREAMLINED STM ANALYSIS — PARAMETERIZED (MULTI-COVARIATE)")
        print("="*70)
        print(f"Input CSV            : {input_csv}")
        print(f"Input Mode           : {input_mode}")
        print(f"Topics (n_topics)    : {n_topics}")
        print(f"Top Samples per Topic: {top_samples_count}")
        print(f"Use Covariates       : {use_covariates} (weight={covariate_weight})")
        print("="*70)

    # ---------- Load ----------
    df = pd.read_csv(input_csv, encoding="utf-8")
    n0 = len(df)
    doc_flow = [{"Step": "Loaded", "Documents": n0}]  # document counts at each step
    if verbose: print(f"Loaded: {n0} rows")

    # Ensure required text columns exist
    for c in [col_forum, col_title, col_post]:
        if c not in df.columns:
            df[c] = ""

    # Build combined_text according to input_mode
    t = df[col_title].fillna("").astype(str)
    p = df[col_post].fillna("").astype(str)
    if input_mode == "title_only":
        combined = t
    elif input_mode == "post_only":
        combined = p
    elif input_mode == "title_post":
        combined = (t + " " + p).str.replace(r"\s+", " ", regex=True).str.strip()
    else:
        raise ValueError("input_mode must be one of {'title_only','post_only','title_post'}")

    df[col_combined] = combined
    df = df[df[col_combined].str.strip() != ""].copy().reset_index(drop=True)
    if verbose: print(f"After empty-text removal: {len(df)} rows")
    doc_flow.append({"Step": f"After empty-text removal ({input_mode})", "Documents": len(df)})

 # ---------- Text preprocessing ----------
    stop_words = set(stopwords.words("english"))
    # Edit the custom stop words here if desired
    medical_stopwords = {"patient","doctor","hospital","treatment","hi","hello","thanks","thank","please"}
    stop_words.update(medical_stopwords)
    lemmatizer = nltk.WordNetLemmatizer()

    def preprocess_text(text:str) -> str:
        if not isinstance(text, str): return ""
        text = "".join(ch.lower() for ch in text if (ch.isalpha() or ch.isspace()) and ch.isascii())
        tokens = [w for w in text.split() if len(w) > 2 and w not in stop_words]
        lemmas = [lemmatizer.lemmatize(w) for w in tokens if len(w) > 2]
        return " ".join(lemmas)

    df["text_processed"] = df[col_combined].apply(preprocess_text)
    df = df[df["text_processed"].str.strip() != ""].copy().reset_index(drop=True)
    if verbose: print(f"After preprocessing: {len(df)} rows")
    doc_flow.append({"Step": "After preprocessing (non-empty processed text)", "Documents": len(df)})

    # ---------- Vocabulary ----------
    if verbose: print("Building vocabulary…")
    base_vect = CountVectorizer(
        max_df=0.95,
        min_df=min_word_frequency,
        stop_words="english",
        token_pattern=r"\b[a-zA-Z]{3,}\b"
    )
    base_matrix = base_vect.fit_transform(df["text_processed"])
    base_vocab = base_vect.get_feature_names_out()

    # Score terms (simple informative score); cap by max_doc_vocab_ratio
    term_counts = np.array(base_matrix.sum(axis=0)).ravel()
    total_vocab = len(base_vocab)
    cap = max(1, int(total_vocab * max_doc_vocab_ratio))
    scores = term_counts * np.log(len(df) / np.maximum(1, term_counts))
    top_idx = np.argsort(scores)[-cap:]
    selected_vocab = base_vocab[top_idx]

    final_vect = CountVectorizer(vocabulary=selected_vocab, token_pattern=r"\b[a-zA-Z]{3,}\b")
    X = final_vect.fit_transform(df["text_processed"])
    if verbose:
        print(f"Vocabulary: {len(base_vocab)} \u2192 {len(selected_vocab)}")
        print(f"Document-Term Matrix: {X.shape}")

    # Document and term counts at each vocabulary step
    _tp = r"\b[a-zA-Z]{3,}\b"
    _n_all = len(CountVectorizer(token_pattern=_tp).fit(df["text_processed"]).vocabulary_)
    _m = CountVectorizer(token_pattern=_tp, stop_words="english").fit_transform(df["text_processed"])
    _dfreq = np.asarray((_m > 0).sum(axis=0)).ravel()
    vocab_flow = pd.DataFrame([
        {"Step": "Unique terms after preprocessing", "Terms": _n_all},
        {"Step": "After scikit-learn English stop-word list", "Terms": _m.shape[1]},
        {"Step": f"Removed: in fewer than {min_word_frequency} documents (min_df)",
         "Terms": -int((_dfreq < min_word_frequency).sum())},
        {"Step": "Removed: in more than 95% of documents (max_df=0.95)",
         "Terms": -int(((_dfreq >= min_word_frequency) & (_dfreq > 0.95 * len(df))).sum())},
        {"Step": "After frequency thresholds", "Terms": int(total_vocab)},
        {"Step": f"Final vocabulary: top {max_doc_vocab_ratio:.0%} by total_count*log(N_docs/total_count)",
         "Terms": int(len(selected_vocab))},
    ])
    doc_len = np.asarray(X.sum(axis=1)).ravel()
    doc_flow.append({"Step": "Documents with >= 1 term in final vocabulary (0-term documents kept in LDA, "
                             "excluded from effect estimation)", "Documents": int((doc_len > 0).sum())})
    doc_flow = pd.DataFrame(doc_flow)
    if verbose:
        print(vocab_flow.to_string(index=False)); print(doc_flow.to_string(index=False))

    # ---------- Covariates (Build X_cov and cov_meta) ----------
    X_cov = None
    cov_meta = {"categorical": [], "numeric": [], "expanded_map": {}} # Initialize cov_meta
    if use_covariates and covariate_cols:
        if verbose: print("Building covariate matrix…")
        X_cov, cov_cols, cov_meta = build_covariate_matrix(
            df, # Pass the main DataFrame `df`
            covariate_cols=covariate_cols,
            scale_numeric=scale_numeric_covariates
        )
        if verbose and X_cov is not None:
            print(f"Covariate matrix: {X_cov.shape} features: {cov_cols}")


    # ---------- Model ----------
    if verbose: print("Training STM model…")
    stm = StreamlinedSTM(
        n_topics=n_topics, covariate_weight=covariate_weight,
        alpha=topic_concentration, beta=word_concentration,
        random_state=random_state
    )
    covariates = X_cov if (use_covariates and X_cov is not None) else None
    stm.fit(X, covariates, use_covariates)

    df["Topic"] = stm.doc_topic_dist_.argmax(axis=1)
    feature_names = final_vect.get_feature_names_out()
    topic_words = stm.get_topic_words(feature_names, n_words=50)

    # ---------- Representatives ----------
    def get_representative_samples(topic_df: pd.DataFrame, topic_id:int, k:int) -> pd.DataFrame:
        if len(topic_df) == 0: return pd.DataFrame()
        pos_idx = topic_df.index.to_numpy()
        probs = stm.doc_topic_dist_[pos_idx, topic_id]
        text_len = topic_df["text_processed"].str.len().fillna(0).to_numpy()
        # normalize
        p = (probs - probs.min()) / (probs.max() - probs.min() + 1e-8)
        q = (text_len - text_len.min()) / (text_len.max() - text_len.min() + 1e-8)
        score = 0.7 * p + 0.3 * q
        n = min(int(k), len(topic_df))
        top_loc = np.argsort(score)[-n:][::-1]
        reps = topic_df.iloc[top_loc].copy()
        reps["Topic_Probability"] = probs[top_loc]
        reps["Quality_Score"] = score[top_loc]
        return reps

    # ---------- Metrics ----------
    perplexity = stm.lda.perplexity(X)
    try:
        sil = silhouette_score(stm.doc_topic_dist_, df["Topic"].values)
    except Exception:
        sil = -1.0

    # Covariate diagnostics
    cov_consistency = 0.0
    cov_diagnostics: Dict[str, float] = {}
    if use_covariates and covariate_cols:
        # Categorical: per-topic dominant-category share → mean across topics → mean across cats
        cat_scores = []
        for c in cov_meta["categorical"]:
            s = df[c].astype("string").fillna("MISSING")
            topic_scores = []
            for t in range(n_topics):
                mask = (df["Topic"] == t).values
                if mask.any():
                    vals, counts = np.unique(s.values[mask], return_counts=True)
                    topic_scores.append(counts.max() / counts.sum())
            if topic_scores:
                cat_scores.append(float(np.mean(topic_scores)))
        cat_score = float(np.mean(cat_scores)) if cat_scores else 0.0

        # Numeric: ANOVA-style eta-squared across topics
        num_scores = []
        for c in cov_meta["numeric"]:
            v = pd.to_numeric(df[c], errors="coerce").to_numpy()
            if not np.any(np.isfinite(v)):
                continue
            m = np.nanmean(v)
            v = np.where(np.isfinite(v), v, m)
            grand_mean = float(np.mean(v))
            ss_between = 0.0
            for t in range(n_topics):
                mask = (df["Topic"] == t)
                if mask.any():
                    vt = v[mask]
                    ss_between += len(vt) * (float(np.mean(vt)) - grand_mean) ** 2
            ss_total = float(np.sum((v - grand_mean) ** 2)) or 1.0
            eta2 = ss_between / ss_total
            num_scores.append(float(eta2))
        num_score = float(np.mean(num_scores)) if num_scores else 0.0

        parts = [p for p in [cat_score, num_score] if np.isfinite(p)]
        cov_consistency = float(np.mean(parts)) if parts else 0.0
        cov_diagnostics = {
            "categorical_topic_purity_mean": cat_score,
            "numeric_eta2_mean": num_score
        }

    topic_div = np.mean([entropy(comp / comp.sum()) for comp in stm.components_])

    metrics = {
        "perplexity": float(perplexity),
        "silhouette": float(sil),
        "covariate_consistency": float(cov_consistency),
        "topic_diversity": float(topic_div),
        "documents": int(len(df)),
        "vocab_size": int(len(feature_names)),
        **cov_diagnostics
    }
    if verbose:
        print(f"Perplexity: {perplexity:.2f} | Silhouette: {sil:.4f}")
        print(f"Covariate Consistency: {cov_consistency:.4f} | Topic Diversity: {topic_div:.4f}")

    # ---------- Tables (Excel sheets) ----------
    if verbose: print("Collecting tables…")
    sheets: Dict[str, pd.DataFrame] = {}   # sheet name -> table (written by the export cell)
    # Config
    sheets["Config"] = pd.DataFrame({
        "Parameter": [
            "Model Type","Input Mode","Use Covariates","Covariate Weight",
            "Covariates Used","Scale Numeric Covariates",
            "Min Word Freq","Doc Vocab Ratio","Topic Number",
            "Top Samples Count","Perplexity","Silhouette",
            "Covariate Consistency","Cat Topic Purity (mean)","Numeric \u03b7\u00b2 (mean)",
            "Topic Diversity","Total Documents","Vocabulary Size"
        ],
        "Value": [
            "Streamlined STM (multi-covariate)", input_mode, use_covariates, covariate_weight,
            ", ".join(covariate_cols) if covariate_cols else "None", scale_numeric_covariates,
            min_word_frequency, max_doc_vocab_ratio, n_topics,
            top_samples_count, f"{perplexity:.2f}", f"{sil:.4f}",
            f"{cov_consistency:.4f}",
            f"{metrics.get('categorical_topic_purity_mean', 0.0):.4f}",
            f"{metrics.get('numeric_eta2_mean', 0.0):.4f}",
            f"{topic_div:.4f}", len(df), len(feature_names)
        ]
    })

    # full model specification, document and term counts
    _versions = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                 "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
                 "nltk": nltk.__version__, "matplotlib": matplotlib.__version__}
    _lp = stm.lda.get_params()
    sheets["Model_Specification"] = pd.DataFrame([
        ("Software", "; ".join(f"{k} {v}" for k, v in _versions.items())),
        ("Text field", input_mode),
        ("Character filter", "lower-case; keep ASCII letters and whitespace only"),
        ("Tokenisation", "whitespace split; tokens of <= 2 characters removed"),
        ("Stop words (1)", f"NLTK English ({len(stopwords.words('english'))} words) + custom: "
                           f"{', '.join(sorted(medical_stopwords))}"),
        ("Lemmatisation", "NLTK WordNetLemmatizer (default noun POS)"),
        ("Stop words (2)", "scikit-learn ENGLISH_STOP_WORDS (318 words), in CountVectorizer"),
        ("Document-term matrix", "unigram raw counts, CountVectorizer, token pattern [a-zA-Z]{3,}"),
        ("min_df / max_df", f"{min_word_frequency} documents / 0.95 of documents"),
        ("Vocabulary cap", f"top {max_doc_vocab_ratio:.0%} of terms by total_count*log(N_docs/total_count)"),
        ("Final DTM", f"{X.shape[0]} documents x {X.shape[1]} terms, {int(X.sum())} tokens"),
        ("LDA implementation", "sklearn LatentDirichletAllocation, variational Bayes"),
        ("LDA settings", ", ".join(f"{k}={_lp[k]}" for k in ["n_components", "doc_topic_prior",
            "topic_word_prior", "learning_method", "max_iter", "max_doc_update_iter",
            "mean_change_tol", "evaluate_every", "random_state"])),
        ("Covariate adjustment", f"use_covariates={use_covariates}; covariates="
            f"{', '.join(covariate_cols) if covariate_cols else 'None'}; one-vs-rest logistic "
            f"regression of dominant LDA topic on covariates; theta_adj=(1-w)*theta + w*P(topic|cov), "
            f"w={covariate_weight}; numeric covariates z-scored={scale_numeric_covariates}"),
        ("Random seed", random_state),
    ], columns=["Item", "Value"])
    sheets["Document_Flow"] = doc_flow
    sheets["Vocabulary_Flow"] = vocab_flow

    # Topic summary
    sheets["Topic_Summary"] = pd.DataFrame([
        {
            "Topic": t,
            "Top_Keywords": ", ".join([w for w in (topic_words.get(t, [])[:10])]),
            "Document_Count": int((df["Topic"] == t).sum()),
            "Percentage": float((df["Topic"] == t).mean() * 100),
        }
        for t in range(n_topics)
    ])

    # Per-topic reps
    for t in range(n_topics):
        tdf = df[df["Topic"] == t].copy()
        if tdf.empty: continue
        reps = get_representative_samples(tdf, t, top_samples_count).sort_values(
            ["Topic_Probability","Quality_Score"], ascending=False
        )
        cols = [
            col_forum, "date_only", "year",  # app and date for qualitative interpretation
            col_post_type, col_title, col_post, col_combined,
            "text_processed","Topic_Probability","Quality_Score"
        ]
        cols = [c for c in cols if c in reps.columns]
        sheet = f"Topic_{t}" if len(f"Topic_{t}") <= 31 else f"T{t}"
        sheets[sheet] = reps[cols]

    # All docs
    all_df = df[[c for c in [col_forum, "date_only", "year", col_post_type, col_title, col_post,
                             col_combined, "text_processed", "Topic"] if c in df.columns]].copy()
    all_df["Topic_Probability"] = [stm.doc_topic_dist_[i, t] for i, t in enumerate(df["Topic"])]
    sheets["All_Documents"] = all_df
    if output_xlsx:   # only when a file name is given
        with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
            for name, tab in sheets.items():
                tab.to_excel(writer, sheet_name=name, index=False)

# ---------- Heatmap ----------
    if verbose: print("Drawing heatmap…")
    heatmap_fig = None
    # choose heatmap source: explicit > forum > first categorical > None
    heatmap_cov = heatmap_by
    if heatmap_cov is None:
        if col_forum in df.columns and df[col_forum].notna().any():
            heatmap_cov = col_forum
        elif cov_meta.get("categorical"):
            heatmap_cov = cov_meta["categorical"][0]

    if heatmap_cov and (heatmap_cov in df.columns):
        mat = pd.crosstab(df[heatmap_cov].astype("string").fillna("MISSING"), df["Topic"])
        heatmap_fig = plt.figure(figsize=(12, 6))  # figure kept for the combined PDF
        plt.imshow(mat.values, cmap="YlOrRd", aspect="auto")
        plt.colorbar(label="Document Count")
        plt.title(f"{heatmap_cov} vs Topic Distribution\n({n_topics} topics, {len(df)} documents)")
        plt.xlabel("Topic"); plt.ylabel(heatmap_cov)
        plt.xticks(range(n_topics), [f"T{i}" for i in range(n_topics)])
        plt.yticks(range(len(mat.index)), mat.index)
        for i in range(len(mat.index)):
            for j in range(n_topics):
                plt.text(j, i, str(mat.values[i, j]), ha="center", va="center", fontsize=8)
        plt.tight_layout()
        if output_heatmap_png:   # only when a file name is given
            plt.savefig(output_heatmap_png, dpi=300, bbox_inches="tight")
        plt.close()
    else:
        if verbose:
            print("No suitable categorical covariate found for heatmap; skipping.")

    if verbose:
        print("="*70)
        print("COMPLETED")
    # Return artifacts
    out_df = df.copy()
    out_df["Topic_Probability"] = [stm.doc_topic_dist_[i, t] for i, t in enumerate(df["Topic"])]
    return {
        "excel": str(Path(output_xlsx).resolve()) if output_xlsx else None,
        "heatmap": str(Path(output_heatmap_png).resolve()) if output_heatmap_png else None,
        "sheets": sheets,                                          # tables for the export cell
        "figures": [("Topic counts by " + str(heatmap_cov), heatmap_fig)] if heatmap_fig is not None else [],
        "df": out_df,
        "topic_words": topic_words,
        "metrics": metrics,
        # needed by estimate_prevalence_effects()
        "theta_lda": stm.lda_doc_topic_,
        "doc_lengths": doc_len,
        "doc_topic_prior": topic_concentration,
        "document_flow": doc_flow,
        "vocabulary_flow": vocab_flow,
        # needed by select_k()
        "dtm": X,
        "vocab": feature_names,
    }

### Main model


In [ ]:
# Settings of the main topic model, defined once and reused by every later analysis
MAIN_SETTINGS = dict(
    input_mode="post_only",
    n_topics=15,
    min_word_frequency=5,
    max_doc_vocab_ratio=0.05,
    use_covariates=True,
    covariate_weight=0.4,
    topic_concentration=0.1,
    word_concentration=0.01,
    top_samples_count=10,
    random_state=42,

    # MULTI-COVARIATE: mix categorical + numeric (present in your CSV)
    covariate_cols=["forum", "year"],  # App + Year
    heatmap_by="forum",                 # choose which categorical to visualize
    scale_numeric_covariates=True,

    # Column names
    col_forum="forum",
    col_title="original_title",
    col_post="original_post",
    col_combined="combined_text",
    col_post_type="post_type",
)

res = run_stm_analysis(
    input_csv="reshaped.csv",
    output_xlsx=None,          # tables and figures are written by the export cell
    output_heatmap_png=None,
    **MAIN_SETTINGS,
    verbose=True,
)


## App and Year as topic-prevalence covariates
Effects are estimated from the **unadjusted** LDA proportions, because the covariate-adjusted proportions already contain App/Year information.

In [ ]:
import re
# App and Year as topic-prevalence covariates: effects, 95% intervals, tables and plots
# Method of composition, as in R stm::estimateEffect (Treier & Jackman 2008; Roberts et al. 2019):
# repeatedly draw each document's topic proportions from its LDA posterior, regress them on
# App + factor(Year), and pool the coefficient draws. Uses the UNADJUSTED LDA proportions, because
# the covariate-adjusted proportions already contain App/Year information (circular otherwise).
import patsy
from scipy import stats


def _bh(p):
    p = np.asarray(p, float); n = len(p)
    if n == 0: return p
    o = np.argsort(p); q = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    out = np.empty(n); out[o] = np.clip(q, 0, 1); return out


def _summ(d, level=0.95):
    a = (1 - level) / 2
    return {"est": d.mean(0), "se": d.std(0, ddof=1),
            "lo": np.quantile(d, a, axis=0), "hi": np.quantile(d, 1 - a, axis=0)}


def estimate_prevalence_effects(
    res, *, app_col="forum", year_col="year", reference_app=None,
    year_as="categorical",        # "categorical" | "linear" | "spline" (sensitivity analyses)
    interaction=False,            # True adds App x Year
    exclude_apps=(),              # optional: applications to leave out of the effect estimation
    weight_apps_equally=False,    # weighted least squares: every application contributes equally
    n_sims=200, ci_level=0.95, fdr_alpha=0.05, min_effect_pp=2.0,
    random_state=42, out_prefix=None, verbose=True):
    # out_prefix=None writes no files: tables are returned in ["sheets"] and figures in ["figures"].
    # With a single application (e.g. Replika only) the App term is dropped: theta_k ~ factor(Year).

    df = res["df"].reset_index(drop=True)
    theta, doc_len, alpha = res["theta_lda"], res["doc_lengths"], res["doc_topic_prior"]
    K = theta.shape[1]
    labels = [f"T{k}: " + ", ".join(res["topic_words"][k][:4]) for k in range(K)]

    # Sample for effect estimation
    flow = [("Documents in topic model", len(df))]
    keep = pd.to_numeric(df[year_col], errors="coerce").notna().to_numpy().copy()
    flow.append(("With valid year", int(keep.sum())))
    keep &= df[app_col].notna().to_numpy() & (df[app_col].astype(str).str.strip() != "").to_numpy()
    flow.append(("With valid year and App", int(keep.sum())))
    keep &= doc_len > 0
    flow.append(("... and >= 1 term in final vocabulary", int(keep.sum())))
    if exclude_apps:
        keep &= ~df[app_col].astype(str).str.strip().isin(list(exclude_apps)).to_numpy()
        flow.append((f"... excluding {', '.join(exclude_apps)} (used)", int(keep.sum())))
    flow = pd.DataFrame(flow, columns=["Step", "Documents"])

    meta = pd.DataFrame({"App": df.loc[keep, app_col].astype(str).str.strip().values,
                         "year_num": pd.to_numeric(df.loc[keep, year_col]).astype(int).values})
    meta["year"] = meta["year_num"].astype(str)
    # Variational Dirichlet parameters: gamma_d = theta_d * (K*alpha + N_d)
    gamma = theta[keep] * (K * alpha + doc_len[keep])[:, None]

    apps = meta["App"].value_counts().index.tolist()
    ref_app = reference_app or apps[0]
    apps = [ref_app] + sorted(a for a in apps if a != ref_app)
    years = sorted(meta["year_num"].unique().tolist()); ref_year = years[0]
    single_app = len(apps) == 1   # e.g. the Replika-only analysis
    app_term = f"C(App, Treatment(reference={ref_app!r}))"
    year_term = {"categorical": f"C(year, Treatment(reference={str(ref_year)!r}))",
                 "linear": "year_num", "spline": "bs(year_num, df=3)"}[year_as]
    formula = f"~ {app_term} {'*' if interaction else '+'} {year_term}"
    readable = (f"theta_k ~ App [ref {ref_app}] {'*' if interaction else '+'} "
                f"{ {'categorical': f'factor(Year) [ref {ref_year}]', 'linear': 'Year', 'spline': 'B-spline(Year, df=3)'}[year_as] }")
    if single_app:   # App is constant: Year only
        formula = f"~ {year_term}"
        readable = (f"theta_k ~ {({'categorical': f'factor(Year) [ref {ref_year}]', 'linear': 'Year', 'spline': 'B-spline(Year, df=3)'}[year_as])}"
                    f"   (single application: {ref_app})")
    cells = pd.crosstab(meta["App"], meta["year_num"])
    if verbose:
        print(flow.to_string(index=False)); print("Model:", readable); print(cells)
        if (cells < 30).values.any(): print("⚠ Some App x Year cells have < 30 documents (wide intervals).")
    figures = []   # (title, figure) pairs for the combined PDF

    # Method of composition
    Xdf = patsy.dmatrix(formula, meta, return_type="dataframe"); Xd = Xdf.values; dinfo = Xdf.design_info
    n, p = Xd.shape
    if np.linalg.matrix_rank(Xd) < p:
        raise ValueError("Design matrix rank-deficient (empty App x Year cell?). Use interaction=False.")
    # observation weights: 1 = ordinary least squares; equal weighting gives every app the same total weight
    wts = np.ones(n)
    if weight_apps_equally and not single_app:
        wts = 1.0 / meta["App"].map(meta["App"].value_counts()).to_numpy(float)
        wts = wts * n / wts.sum()
    XtW = Xd.T * wts
    XtX_inv = np.linalg.inv(XtW @ Xd); w, V = np.linalg.eigh(XtX_inv)
    L = V * np.sqrt(np.clip(w, 0, None)); proj = XtX_inv @ XtW
    rng = np.random.default_rng(random_state); draws = np.empty((n_sims, p, K))
    for m in range(n_sims):
        g = rng.standard_gamma(gamma); th = g / g.sum(1, keepdims=True)
        B = proj @ th; s2 = (wts[:, None] * (th - Xd @ B) ** 2).sum(0) / (n - p)
        draws[m] = B + (L @ rng.standard_normal((p, K))) * np.sqrt(s2)[None, :]

    # Coefficients
    names = [re.sub(r"C\((\w+), Treatment\(reference='[^']*'\)\)\[T\.([^\]]*)\]", r"\1 = \2", c)
             for c in Xdf.columns]
    s = _summ(draws, ci_level)
    coef = pd.DataFrame([{"Topic": k, "Topic_label": labels[k], "Term": names[j],
                          "Estimate": s["est"][j, k], "SE": s["se"][j, k],
                          "CI_low": s["lo"][j, k], "CI_high": s["hi"][j, k],
                          "p_approx": 2 * stats.norm.sf(abs(s["est"][j, k] / s["se"][j, k]))}
                         for k in range(K) for j in range(p)])

    # Joint Wald tests per model term and topic
    wald = []
    for tname, sl in dinfo.term_name_slices.items():
        if tname == "Intercept": continue
        lab = "App x Year" if ("App" in tname and "year" in tname) else ("App" if "App" in tname else "Year")
        for k in range(K):
            d = draws[:, sl, k]; b = d.mean(0); S = np.atleast_2d(np.cov(d, rowvar=False))
            W = float(b @ np.linalg.pinv(S) @ b); dof = int(np.linalg.matrix_rank(S))
            wald.append({"Topic": k, "Topic_label": labels[k], "Term": lab, "Wald_chi2": W, "df": dof,
                         "p": stats.chi2.sf(W, dof)})
    wald = pd.DataFrame(wald); wald["q_BH"] = wald.groupby("Term")["p"].transform(lambda x: _bh(x.values))

    # Expected proportions: everyone set to one App (years as observed) / one Year (apps as observed)
    def cf(set_cols):
        c = meta.copy()
        for kk, vv in set_cols.items(): c[kk] = vv
        Xc = patsy.build_design_matrices([dinfo], c, return_type="dataframe")[0].values
        return np.einsum("p,mpk->mk", (Xc * wts[:, None]).sum(0) / wts.sum(), draws)
    pa = {a: cf({"App": a}) for a in apps}
    py = {y: cf({"year_num": y, "year": str(y)}) for y in years}

    def ptab(pred, by):
        rows = []
        for lev, d in pred.items():
            sm = _summ(d, ci_level)
            rows += [{"Topic": k, "Topic_label": labels[k], by: lev, "Expected_proportion": sm["est"][k],
                      "SE": sm["se"][k], "CI_low": sm["lo"][k], "CI_high": sm["hi"][k]} for k in range(K)]
        return pd.DataFrame(rows)
    pred_app, pred_year = ptab(pa, "App"), ptab(py, "Year")

    def ctab(family, pairs, pred):
        rows = []
        for a, b in pairs:
            sm = _summ(pred[a] - pred[b], ci_level)
            rows += [{"Family": family, "Topic": k, "Topic_label": labels[k], "Contrast": f"{a} - {b}",
                      "Level": a, "Baseline": b, "Difference_pp": 100 * sm["est"][k], "SE_pp": 100 * sm["se"][k],
                      "CI_low_pp": 100 * sm["lo"][k], "CI_high_pp": 100 * sm["hi"][k],
                      "p_approx": 2 * stats.norm.sf(abs(sm["est"][k] / sm["se"][k]))} for k in range(K)]
        return rows
    contr = pd.DataFrame(
        ctab("App (pairwise)", [(apps[j], apps[i]) for i in range(len(apps)) for j in range(i + 1, len(apps))], pa)
        + ctab(f"Year (vs {ref_year})", [(y, ref_year) for y in years if y != ref_year], py))
    contr["q_BH"] = contr.groupby("Family")["p_approx"].transform(lambda x: _bh(x.values))
    contr["Meaningful"] = (((contr.CI_low_pp > 0) | (contr.CI_high_pp < 0)) & (contr.q_BH < fdr_alpha)
                           & (contr.Difference_pp.abs() >= min_effect_pp))
    meaningful = (contr[contr.Meaningful].assign(_a=lambda x: x.Difference_pp.abs())
                  .sort_values(["Family", "_a"], ascending=[True, False]).drop(columns="_a"))

    spec = pd.DataFrame([
        ("Prevalence model", readable), ("patsy formula", formula),
        ("Excluded applications", ", ".join(exclude_apps) or "none"),
        ("Weighting", "each application weighted equally (WLS)" if (weight_apps_equally and not single_app) else "none (OLS)"),
        ("Documents / topics", f"{n} / {K}"),
        ("Proportions used", "unadjusted LDA posterior (before covariate adjustment)"),
        ("Estimation", "OLS per topic; method of composition with draws from each document's "
                       "variational Dirichlet posterior (as stm::estimateEffect, uncertainty='Global')"),
        ("Simulations / seed", f"{n_sims} / {random_state}"),
        ("Intervals", f"{ci_level:.0%} percentile intervals of pooled draws"),
        ("Expected proportions", "average prediction with all documents set to one App (years as observed) "
                                 "or one Year (apps as observed)" if not single_app else
                                 "average prediction with all documents set to one Year"),
        ("Multiple testing", f"Benjamini-Hochberg within contrast family, FDR {fdr_alpha}"),
        ("Meaningful difference", f"{ci_level:.0%} CI excludes 0 AND q < {fdr_alpha} AND |diff| >= {min_effect_pp} pp"),
        ("Packages", f"patsy {patsy.__version__}; scipy {scipy.__version__}; numpy {np.__version__}"),
    ], columns=["Item", "Value"])

    # ---------- Plots ----------
    ncol = 4; nrow = int(np.ceil(K / ncol)); col = {a: plt.get_cmap("tab10")(i % 10) for i, a in enumerate(apps)}
    if not single_app:   # no App plot when there is a single application
        fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.8 * nrow), squeeze=False)
        for k, ax in enumerate(axes.flat):
            if k >= K: ax.axis("off"); continue
            sub = pred_app[pred_app.Topic == k].set_index("App").loc[apps]
            for i, a in enumerate(apps):
                r = sub.loc[a]
                ax.errorbar(i, 100 * r.Expected_proportion, yerr=[[100 * (r.Expected_proportion - r.CI_low)],
                            [100 * (r.CI_high - r.Expected_proportion)]], fmt="o", color=col[a], capsize=3)
            ax.set_xticks(range(len(apps))); ax.set_xticklabels(apps, rotation=45, ha="right", fontsize=8)
            ax.set_title(labels[k], fontsize=9); ax.set_ylabel("%", fontsize=8)
        fig.suptitle(f"Expected topic prevalence by app ({ci_level:.0%} CI), adjusted for year", y=1.0)
        fig.tight_layout(); figures.append(("Expected topic prevalence by app", fig))
        if out_prefix: fig.savefig(f"{out_prefix}_by_app.png", dpi=300, bbox_inches="tight")
        plt.show()

    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.8 * nrow), squeeze=False)
    for k, ax in enumerate(axes.flat):
        if k >= K: ax.axis("off"); continue
        sub = pred_year[pred_year.Topic == k].sort_values("Year")
        ax.plot(sub.Year, 100 * sub.Expected_proportion, "o-", color="black", ms=3)
        ax.fill_between(sub.Year, 100 * sub.CI_low, 100 * sub.CI_high, color="grey", alpha=0.25)
        ax.set_xticks(years); ax.tick_params(labelsize=7); ax.tick_params(axis="x", rotation=45)
        ax.set_title(labels[k], fontsize=9); ax.set_ylabel("%", fontsize=8)
    fig.suptitle(f"Expected topic prevalence by year ({ci_level:.0%} CI)"
                 + ("" if single_app else ", adjusted for app"), y=1.0)
    fig.tight_layout(); figures.append(("Expected topic prevalence by year", fig))
    if out_prefix: fig.savefig(f"{out_prefix}_by_year.png", dpi=300, bbox_inches="tight")
    plt.show()

    ac = contr[contr.Family == "App (pairwise)"]; cons = list(dict.fromkeys(ac.Contrast))
    if cons:
        fig, axes = plt.subplots(1, len(cons), figsize=(4.2 * len(cons), 0.35 * K + 1.5), sharey=True, squeeze=False)
        for ax, c in zip(axes.flat, cons):
            sub = ac[ac.Contrast == c].sort_values("Topic")
            for yi, r in zip(np.arange(K)[::-1], sub.itertuples()):
                ax.plot([r.CI_low_pp, r.CI_high_pp], [yi, yi], color="black", lw=1)
                ax.plot(r.Difference_pp, yi, "o", color="black", mfc="black" if r.Meaningful else "white")
            ax.axvline(0, color="red", ls="--", lw=0.8); ax.set_title(c, fontsize=10)
            ax.set_xlabel("Difference (percentage points)")
        axes.flat[0].set_yticks(np.arange(K)[::-1]); axes.flat[0].set_yticklabels(labels, fontsize=8)
        fig.suptitle("Pairwise app differences in topic prevalence (filled = meaningful)", y=1.0)
        fig.tight_layout(); figures.append(("Pairwise app differences", fig))
        if out_prefix: fig.savefig(f"{out_prefix}_app_contrasts.png", dpi=300, bbox_inches="tight")
        plt.show()

    # ---------- Tables ----------
    # tables collected in a dict; a file is written only when out_prefix is given
    sheets = {"Effect_Specification": spec, "Sample": flow,
              "Docs_App_x_Year": cells.reset_index().rename(columns={"index": "App"}),
              "Joint_Tests": wald, "Coefficients": coef}
    if not single_app:
        sheets["Expected_by_App"] = pred_app
    sheets.update({"Expected_by_Year": pred_year, "Contrasts": contr, "Meaningful_Differences": meaningful})
    if out_prefix:
        with pd.ExcelWriter(f"{out_prefix}.xlsx", engine="openpyxl") as wr:
            for name, tab in sheets.items():
                tab.to_excel(wr, sheet_name=name, index=False)

    if verbose:
        print("\nTopics with a significant joint effect (BH q < %.2f):" % fdr_alpha)
        print(wald[wald.q_BH < fdr_alpha].groupby("Term").Topic.count().to_string())
        print(f"\nMeaningful differences: {len(meaningful)}")
        for r in meaningful.head(30).itertuples():
            print(f"  {r.Topic_label}: {r.Difference_pp:+.1f} pp, {r.Level} vs {r.Baseline} "
                  f"(95% CI {r.CI_low_pp:.1f} to {r.CI_high_pp:.1f}; q={r.q_BH:.2g})")
        if out_prefix: print(f"\nWrote {out_prefix}.xlsx and {out_prefix}_*.png")
    return {"specification": spec, "sample": flow, "coefficients": coef, "joint_tests": wald,
            "expected_by_app": pred_app, "expected_by_year": pred_year, "contrasts": contr,
            "meaningful": meaningful, "excel": f"{out_prefix}.xlsx" if out_prefix else None,
            "sheets": sheets, "figures": figures, "single_app": single_app}

In [ ]:
# Run the effect estimation (App + Year as topic-prevalence covariates)
eff = estimate_prevalence_effects(
    res,
    app_col="forum", year_col="year",
    reference_app="Replika",   # reference application for the App coefficients
    year_as="categorical",     # "linear" or "spline" for sensitivity analyses
    interaction=False,         # True adds App x Year
    n_sims=200, ci_level=0.95, fdr_alpha=0.05, min_effect_pp=2.0,
    random_state=42,
)


## Choosing the number of topics
Held-out perplexity, semantic coherence, exclusivity and topic diversity for a range of K, as in `stm::searchK`.

In [ ]:
# Diagnostics for choosing the number of topics K
# For each candidate K, LDA (same settings as the main model, no covariate adjustment) is fitted
# on a random 80% of documents and evaluated with the standard diagnostics used in stm::searchK:
#   held-out perplexity  : perplexity on the remaining 20% of documents (lower = better fit)
#   semantic coherence   : UMass coherence of each topic's top 10 words (Mimno et al. 2011), as in
#                          stm::semanticCoherence (higher / closer to 0 = more coherent)
#   exclusivity          : FREX-based exclusivity of the top 10 words, as in stm::exclusivity
#                          (higher = topics more distinct from each other)
#   topic diversity      : share of unique words among all topics' top 25 words (Dieng et al. 2020)
# Repeating with several seeds shows how stable each diagnostic is.
from sklearn.decomposition import LatentDirichletAllocation
from scipy import stats


def _umass_coherence(phi, B, n_top=10):
    """phi: K x V topic-word matrix; B: binary docs x V (scipy sparse)."""
    df_ = np.asarray(B.sum(axis=0)).ravel()
    co = (B.T @ B).toarray()
    out = []
    for k in range(phi.shape[0]):
        top = np.argsort(phi[k])[::-1][:n_top]
        s = 0.0
        for i in range(1, len(top)):
            for j in range(i):
                s += np.log((co[top[i], top[j]] + 1) / df_[top[j]])
        out.append(s)
    return np.array(out)


def _exclusivity(phi, n_top=10, frexw=0.7):
    """FREX exclusivity as implemented in stm::exclusivity."""
    V = phi.shape[1]
    tb = phi.T                                   # V x K
    mat = tb / tb.sum(axis=1, keepdims=True)
    ex = np.apply_along_axis(lambda x: stats.rankdata(x), 0, mat) / V
    fr = np.apply_along_axis(lambda x: stats.rankdata(x), 0, tb) / V
    frex = 1 / (frexw / ex + (1 - frexw) / fr)
    return np.array([frex[np.argsort(tb[:, k])[::-1][:n_top], k].sum() for k in range(tb.shape[1])])


def _diversity(phi, n_top=25):
    top = [w for k in range(phi.shape[0]) for w in np.argsort(phi[k])[::-1][:n_top]]
    return len(set(top)) / len(top)


def select_k(res, K_values=(5, 10, 15, 20, 25, 30), seeds=(42, 1, 2), test_share=0.2,
             alpha=0.1, eta=0.01, max_iter=50, chosen_k=None, out_prefix=None):
    # out_prefix=None writes no files; returns (summary, runs, figures)
    X = res["dtm"]
    X = X[np.where(np.asarray(X.sum(axis=1)).ravel() > 0)[0]]   # documents with >= 1 term
    B = (X > 0).astype(float)
    rows = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        test = rng.random(X.shape[0]) < test_share
        Xtr, Xte = X[np.where(~test)[0]], X[np.where(test)[0]]
        for K in K_values:
            lda = LatentDirichletAllocation(n_components=K, doc_topic_prior=alpha, topic_word_prior=eta,
                                            max_iter=max_iter, learning_method="batch", random_state=seed)
            lda.fit(Xtr)
            phi = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
            rows.append({"K": K, "seed": seed,
                         "heldout_perplexity": lda.perplexity(Xte),
                         "semantic_coherence": _umass_coherence(phi, B).mean(),
                         "exclusivity": _exclusivity(phi).mean(),
                         "topic_diversity": _diversity(phi)})
            print(f"K={K:>3} seed={seed:>3}  perplexity={rows[-1]['heldout_perplexity']:.1f}  "
                  f"coherence={rows[-1]['semantic_coherence']:.2f}  exclusivity={rows[-1]['exclusivity']:.2f}  "
                  f"diversity={rows[-1]['topic_diversity']:.3f}")
    runs = pd.DataFrame(rows)
    summ = runs.groupby("K").agg(["mean", "std"]).drop(columns="seed")
    summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    summ = summ.reset_index()

    metrics = [("heldout_perplexity", "Held-out perplexity (lower = better)"),
               ("semantic_coherence", "Semantic coherence (higher = better)"),
               ("exclusivity", "Exclusivity (higher = better)"),
               ("topic_diversity", "Topic diversity (higher = better)")]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
    for ax, (m, t) in zip(axes, metrics):
        ax.errorbar(summ.K, summ[f"{m}_mean"], yerr=summ[f"{m}_std"], fmt="o-", color="black", capsize=3)
        if chosen_k is not None:
            ax.axvline(chosen_k, color="red", ls="--", lw=0.8)
        ax.set_title(t, fontsize=10); ax.set_xlabel("Number of topics K"); ax.set_xticks(list(K_values))
    fig.suptitle("Diagnostics by number of topics (mean ± SD over seeds)", y=1.04)
    fig.tight_layout(); figures = [("Diagnostics by number of topics", fig)]
    if out_prefix: fig.savefig(f"{out_prefix}_diagnostics.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(summ.semantic_coherence_mean, summ.exclusivity_mean, color="black")
    for _, r in summ.iterrows():
        ax.annotate(f"K={int(r.K)}", (r.semantic_coherence_mean, r.exclusivity_mean),
                    textcoords="offset points", xytext=(5, 3), fontsize=9,
                    color="red" if chosen_k == r.K else "black")
    ax.set_xlabel("Semantic coherence"); ax.set_ylabel("Exclusivity")
    ax.set_title("Coherence vs exclusivity by K")
    fig.tight_layout(); figures.append(("Coherence vs exclusivity by K", fig))
    if out_prefix: fig.savefig(f"{out_prefix}_coherence_exclusivity.png", dpi=300, bbox_inches="tight")
    plt.show()

    if out_prefix:
        with pd.ExcelWriter(f"{out_prefix}.xlsx", engine="openpyxl") as w:
            summ.to_excel(w, sheet_name="Summary", index=False)
            runs.to_excel(w, sheet_name="All_runs", index=False)
        print(f"\nWrote {out_prefix}.xlsx and {out_prefix}_*.png")
    print(summ.round(3).to_string(index=False))
    return summ, runs, figures

In [ ]:
# Run the K diagnostics (takes a few minutes; each K is fitted once per seed)
k_summary, k_runs, k_figs = select_k(
    res,
    K_values=(5, 10, 15, 20, 25, 30),
    seeds=(42, 1, 2),
    chosen_k=15,          # the K used in the paper (marked in red)
)


## Sensitivity analyses: unequal numbers of reviews across applications
Run after the main model (`res`) and the main effect analysis (`eff`). Happify and Mindspa are excluded from all analyses (removed when the data are read).

| Analysis | Topic model | Effects | Question it answers |
|---|---|---|---|
| `weighted` | main model (K = 15) | App + Year, every application weighted equally | Are the main effects driven by Replika's size? |
| `excl_replika` | refitted without Replika (K = 10) | App + Year | Do the other applications show the same topics and differences on their own? |
| `balanced_s42`, `_s1`, `_s2` | refitted with Replika down-sampled to the largest other application (K = 10), three random samples | App + Year | Same, keeping Replika but at a comparable size |
| `replika_only` | refitted on Replika only (K = 15, main settings) | Year only | Does Replika on its own show the same topics and trends over time? |

No files are written in this section: everything is collected and written by the export cell at the end.


In [ ]:
# Sensitivity analyses for the unequal number of reviews across applications
#   (1) weighted      : main topic model, effects re-estimated with every application weighted equally
#   (2) excl_replika  : topic model and effects refitted WITHOUT Replika
#   (3) balanced_s*   : Replika randomly down-sampled to the size of the largest other application,
#                       topic model and effects refitted (three random samples)
#   (4) replika_only  : topic model refitted on Replika reviews only; Year effects only
#                       (App is constant), compared with the main model topic by topic
# no files are written here; tables and figures are kept in memory for the export cell.
import os, tempfile

# Smaller corpora (excl_replika, balanced): keep all terms passing min_df / max_df, K = 10 (chosen with select_k)
SENS_SETTINGS = {**MAIN_SETTINGS, "max_doc_vocab_ratio": 1.0, "n_topics": 10}
# Replika-only: about 90% of the main corpus, so the main settings are kept (K = 15, same vocabulary rule).
# App is constant, so Year is the only covariate in the adjustment and the heatmap shows topics by year.
REPLIKA_SETTINGS = {**MAIN_SETTINGS, "covariate_cols": ["year"], "heatmap_by": "year"}


def _topic_overlap(res_a, res_b, n_words=20):
    """For each topic in res_a, the most similar topic in res_b (Jaccard overlap of top-20 words)."""
    rows = []
    for ka, wa in res_a["topic_words"].items():
        sa = set(wa[:n_words])
        best = max(res_b["topic_words"].items(), key=lambda kv: len(sa & set(kv[1][:n_words])))
        sb = set(best[1][:n_words])
        rows.append({"Topic": ka, "Top_words": ", ".join(wa[:8]),
                     "Best_matching_main_topic": best[0], "Main_topic_words": ", ".join(best[1][:8]),
                     "Jaccard_top20": len(sa & sb) / len(sa | sb)})
    return pd.DataFrame(rows)


def run_sensitivity(name, subset_fn, res_main, reference_app, settings, sample_seed=42):
    d = pd.read_csv("reshaped.csv")
    d = d[d["forum"].notna() & pd.to_numeric(d["year"], errors="coerce").notna()]   # app and year known
    d = subset_fn(d, sample_seed)
    print(f"\n=== {name}: {len(d)} reviews ===\n{d['forum'].value_counts().to_string()}")
    # the subset is written to a temporary file (run_stm_analysis reads a CSV) and deleted afterwards
    fd, path = tempfile.mkstemp(suffix=".csv"); os.close(fd)
    try:
        d.to_csv(path, index=False)
        r = run_stm_analysis(input_csv=path, output_xlsx=None, output_heatmap_png=None,
                             **settings, verbose=False)            # LDA seed 42 as in the main model
    finally:
        os.remove(path)
    e = estimate_prevalence_effects(r, app_col="forum", year_col="year", reference_app=reference_app,
                                    n_sims=200, random_state=42, verbose=False)
    overlap = _topic_overlap(r, res_main)
    K = settings["n_topics"]
    print(f"Topics with a clear counterpart in the main model (Jaccard >= 0.3): "
          f"{(overlap.Jaccard_top20 >= 0.3).sum()} of {K}")
    print(f"Meaningful differences: {len(e['meaningful'])}")
    row = {"Analysis": name, "Applications": ", ".join(sorted(d["forum"].unique())),
           "Reviews_input": len(d), "Documents_topic_model": len(r["df"]),
           "Documents_effect_analysis": int(e["sample"]["Documents"].iloc[-1]),
           "Topics_K": K, "Vocabulary_terms": len(r["vocab"]), "Sampling_seed": sample_seed,
           "Effect_model": e["specification"].set_index("Item").loc["Prevalence model", "Value"],
           "Meaningful_differences": len(e["meaningful"]),
           "Topics_matched_to_main_(Jaccard>=0.3)": int((overlap.Jaccard_top20 >= 0.3).sum())}
    return {"row": row, "res": r, "eff": e, "overlap": overlap}


def excl_replika(d, seed):
    return d[d["forum"] != "Replika"]


def balanced(d, seed):
    others = d[d["forum"] != "Replika"]
    cap = others["forum"].value_counts().max()          # size of the largest other application
    rep = d[d["forum"] == "Replika"].sample(n=cap, random_state=seed)
    return pd.concat([others, rep]).sort_index()


def replika_only(d, seed):
    return d[d["forum"] == "Replika"]


In [ ]:
# Run the sensitivity analyses (each refit takes about as long as the main model)
sens = {}

# (1) Main topic model, effects re-estimated with every application weighted equally
eff_w = estimate_prevalence_effects(res, app_col="forum", year_col="year", reference_app="Replika",
                                    weight_apps_equally=True, n_sims=200, random_state=42, verbose=False)

# (2) Without Replika, then check that K = 10 still suits this smaller corpus
#     (the vocabulary does not depend on K, so the diagnostics use this model's document-term matrix)
sens["excl_replika"] = run_sensitivity("excl_replika", excl_replika, res, reference_app="Wysa",
                                       settings=SENS_SETTINGS)
k_er_summary, k_er_runs, k_er_figs = select_k(sens["excl_replika"]["res"], K_values=(5, 8, 10, 12, 15, 20),
                                              seeds=(42, 1, 2), chosen_k=SENS_SETTINGS["n_topics"])

# (3) Replika down-sampled to the largest other application, three random samples
for s in (42, 1, 2):
    sens[f"balanced_s{s}"] = run_sensitivity(f"balanced_s{s}", balanced, res, reference_app="Replika",
                                             settings=SENS_SETTINGS, sample_seed=s)

# (4) Replika only: same settings as the main model; Year is the only prevalence covariate
sens["replika_only"] = run_sensitivity("replika_only", replika_only, res, reference_app="Replika",
                                       settings=REPLIKA_SETTINGS)
k_ro_summary, k_ro_runs, k_ro_figs = select_k(sens["replika_only"]["res"], K_values=(5, 10, 15, 20, 25),
                                              seeds=(42, 1, 2), chosen_k=REPLIKA_SETTINGS["n_topics"])


## Summary, comparisons and export


In [ ]:
# Summary and comparisons with the main analysis
# Sample size and number of topics for every analysis
summary = pd.DataFrame(
    [{"Analysis": "main", "Applications": ", ".join(sorted(res["df"]["forum"].dropna().unique())),
      "Reviews_input": int(res["document_flow"]["Documents"].iloc[0]),
      "Documents_topic_model": len(res["df"]),
      "Documents_effect_analysis": int(eff["sample"]["Documents"].iloc[-1]),
      "Topics_K": res["theta_lda"].shape[1], "Vocabulary_terms": len(res["vocab"]), "Sampling_seed": None,
      "Effect_model": eff["specification"].set_index("Item").loc["Prevalence model", "Value"],
      "Meaningful_differences": len(eff["meaningful"]), "Topics_matched_to_main_(Jaccard>=0.3)": None},
     {"Analysis": "weighted", "Applications": "as main (topic model of the main analysis)",
      "Reviews_input": None, "Documents_topic_model": len(res["df"]),
      "Documents_effect_analysis": int(eff_w["sample"]["Documents"].iloc[-1]),
      "Topics_K": res["theta_lda"].shape[1], "Vocabulary_terms": len(res["vocab"]), "Sampling_seed": None,
      "Effect_model": eff_w["specification"].set_index("Item").loc["Prevalence model", "Value"]
                      + " (applications weighted equally)",
      "Meaningful_differences": len(eff_w["meaningful"]), "Topics_matched_to_main_(Jaccard>=0.3)": None}]
    + [v["row"] for v in sens.values()])

# Same topics, so main and weighted effects can be compared directly
keys = ["Family", "Topic", "Contrast"]
cmp_w = eff["contrasts"][keys + ["Topic_label", "Difference_pp", "Meaningful"]].merge(
    eff_w["contrasts"][keys + ["Difference_pp", "CI_low_pp", "CI_high_pp", "Meaningful"]],
    on=keys, suffixes=("_main", "_weighted"))

# Replika-only vs main: Year effects for each Replika-only topic and its best-matching main topic.
# Main-model year effects are adjusted for App; Replika-only year effects describe Replika alone.
yr = lambda c: c[c["Family"].str.startswith("Year")]
cols = ["Topic", "Topic_label", "Contrast", "Difference_pp", "CI_low_pp", "CI_high_pp", "Meaningful"]
ro = sens["replika_only"]
cmp_ro = (ro["overlap"][["Topic", "Best_matching_main_topic", "Jaccard_top20"]]
          .merge(yr(ro["eff"]["contrasts"])[cols], on="Topic")
          .merge(yr(eff["contrasts"])[cols].rename(columns={"Topic": "Best_matching_main_topic"}),
                 on=["Best_matching_main_topic", "Contrast"], suffixes=("_replika_only", "_main"))
          .rename(columns={"Topic": "Topic_replika_only", "Best_matching_main_topic": "Topic_main"}))
cmp_ro["Clear_topic_match"] = cmp_ro["Jaccard_top20"] >= 0.3
cmp_ro["Same_direction"] = np.sign(cmp_ro["Difference_pp_replika_only"]) == np.sign(cmp_ro["Difference_pp_main"])
cmp_ro["Same_meaningful_classification"] = cmp_ro["Meaningful_replika_only"] == cmp_ro["Meaningful_main"]

print(summary.drop(columns=["Effect_model"]).to_string(index=False))
print(f"\nMain vs weighted: same meaningful/not-meaningful classification for "
      f"{(cmp_w.Meaningful_main == cmp_w.Meaningful_weighted).sum()} of {len(cmp_w)} contrasts")
m = cmp_ro[cmp_ro.Clear_topic_match]
print(f"Replika-only: {ro['row']['Topics_matched_to_main_(Jaccard>=0.3)']} of {REPLIKA_SETTINGS['n_topics']} "
      f"topics have a clear counterpart in the main model")
if len(m):
    print(f"  For these topics, year contrasts: same direction {m.Same_direction.sum()} of {len(m)}, "
          f"same meaningful classification {m.Same_meaningful_classification.sum()} of {len(m)}")


In [ ]:
# Export: everything in three files
#   results_main.xlsx         main topic model, App + Year effects, choice of K
#   results_sensitivity.xlsx  all sensitivity analyses, stacked with an "Analysis" column, and comparisons
#   figures.pdf               every figure, one per page, with a title page before each analysis
from matplotlib.backends.backend_pdf import PdfPages


def _stack(parts):
    """parts: list of (analysis name, DataFrame) -> one DataFrame with an Analysis column first."""
    out = [t.assign(Analysis=a)[["Analysis"] + [c for c in t.columns if c != "Analysis"]]
           for a, t in parts if t is not None and len(t)]
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def _reps(sheets):
    """Topic_<k> sheets of representative reviews -> one long table with a Topic column."""
    return pd.concat([t.assign(Topic=int(n.split("_")[1]))[["Topic"] + list(t.columns)]
                      for n, t in sheets.items() if n.startswith("Topic_") and n[6:].isdigit()],
                     ignore_index=True)


def _write(path, sheets):
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, tab in sheets.items():
            tab.to_excel(w, sheet_name=name[:31], index=False)
    print(f"Wrote {path} ({len(sheets)} sheets, {os.path.getsize(path) / 1e6:.1f} MB)")


# ---------- results_main.xlsx ----------
topic_sheets = {n: t for n, t in res["sheets"].items() if n.startswith("Topic_") and n[6:].isdigit()}
main_sheets = {"README": pd.DataFrame({"Sheet": [
        "Config, Model_Specification", "Document_Flow, Vocabulary_Flow", "Topic_Summary",
        "Effects_*", "K_Selection_*", "Topic_<k>", "All_Documents"], "Contents": [
        "Main topic model settings and fit metrics (Happify and Mindspa excluded from the start)",
        "Number of documents and terms at each preprocessing step",
        f"Top words and size of each of the {MAIN_SETTINGS['n_topics']} topics",
        "App + Year effects on topic prevalence: specification, sample, joint tests, coefficients, "
        "expected proportions, pairwise contrasts, meaningful differences",
        "Diagnostics for the number of topics (held-out perplexity, coherence, exclusivity, diversity)",
        "Representative reviews for each topic, with app and date",
        "Every document with its dominant topic"]})}
main_sheets.update({n: t for n, t in res["sheets"].items() if n not in topic_sheets and n != "All_Documents"})
# Reviews collected and applications removed before the topic model, then the topic-model steps
main_sheets["Document_Flow"] = pd.concat(
    [pd.DataFrame(EXCLUSION_COUNTS, columns=["Step", "Documents"]), res["sheets"]["Document_Flow"]],
    ignore_index=True)
main_sheets.update({f"Effects_{n}": t for n, t in eff["sheets"].items()})
main_sheets.update({"K_Selection_Summary": k_summary, "K_Selection_Runs": k_runs})
main_sheets.update(topic_sheets)
main_sheets["All_Documents"] = res["sheets"]["All_Documents"]
_write("results_main.xlsx", main_sheets)

# ---------- results_sensitivity.xlsx ----------
refits = list(sens.items())                                   # analyses with their own topic model
all_eff = [("weighted", eff_w)] + [(a, v["eff"]) for a, v in refits]
sens_sheets = {"README": pd.DataFrame({"Sheet": [
        "Analyses", "Main_vs_weighted", "ReplikaOnly_vs_main_Year",
        "Topic_Summary, Representative_Reviews, Topic_Match_to_main",
        "Config, Model_Specification, Document_Flow, Vocabulary_Flow",
        "Effect_Specification ... Meaningful_Differences", "K_Selection_Summary, K_Selection_Runs"], "Contents": [
        "One row per analysis: applications, sample sizes, K, vocabulary, effect model, number of meaningful "
        "differences, topics matched to the main model",
        "App and Year contrasts in the main model, unweighted vs every application weighted equally",
        "Replika-only Year contrasts next to the Year contrasts of the best-matching main topic",
        "Topics of each refitted model and their best match in the main model (Jaccard overlap of top-20 words)",
        "Settings and document/term counts of each refitted model",
        "Effect tables, stacked across analyses (Analysis column). Topic numbers refer to each analysis's own "
        "model; weighted uses the main model's topics. replika_only has Year effects only",
        "Diagnostics for the number of topics in excl_replika and replika_only"]})}
sens_sheets["Analyses"] = summary
sens_sheets["Main_vs_weighted"] = cmp_w
sens_sheets["ReplikaOnly_vs_main_Year"] = cmp_ro
sens_sheets["Topic_Summary"] = _stack([(a, v["res"]["sheets"]["Topic_Summary"]) for a, v in refits])
sens_sheets["Representative_Reviews"] = _stack([(a, _reps(v["res"]["sheets"])) for a, v in refits])
sens_sheets["Topic_Match_to_main"] = _stack([(a, v["overlap"]) for a, v in refits])
for n in ["Config", "Model_Specification", "Document_Flow", "Vocabulary_Flow"]:
    sens_sheets[n] = _stack([(a, v["res"]["sheets"][n]) for a, v in refits])
for n in eff["sheets"]:
    sens_sheets[n] = _stack([(a, e["sheets"].get(n)) for a, e in all_eff])
sens_sheets["K_Selection_Summary"] = _stack([("excl_replika", k_er_summary), ("replika_only", k_ro_summary)])
sens_sheets["K_Selection_Runs"] = _stack([("excl_replika", k_er_runs), ("replika_only", k_ro_runs)])
_write("results_sensitivity.xlsx", sens_sheets)

# ---------- figures.pdf ----------
sections = [("Main analysis", res["figures"] + eff["figures"] + k_figs),
            ("Sensitivity: main topics, applications weighted equally", eff_w["figures"])]
for a, v in refits:
    extra = {"excl_replika": k_er_figs, "replika_only": k_ro_figs}.get(a, [])
    sections.append((f"Sensitivity: {a}", v["res"]["figures"] + v["eff"]["figures"] + extra))
n_pages = 0
with PdfPages("figures.pdf") as pdf:
    for title, figs in sections:
        cover = plt.figure(figsize=(11, 2)); cover.text(0.5, 0.5, title, ha="center", va="center", fontsize=20)
        pdf.savefig(cover); plt.close(cover); n_pages += 1
        for _, f in figs:
            pdf.savefig(f, bbox_inches="tight"); n_pages += 1
print(f"Wrote figures.pdf ({n_pages} pages)")

try:   # download in Colab
    from google.colab import files
    for f in ("results_main.xlsx", "results_sensitivity.xlsx", "figures.pdf"):
        files.download(f)
except ImportError:
    pass
